# Semanal 1 · CPU: seleção de variáveis e modelos semanais estatísticos e de árvores
Prevê **1 a 4 semanas à frente** nas 8 temporadas (afinação 2018–21, teste 2022–25) e na rodada do Felipe (2023–24), com o modelo treinado uma vez por rodada.

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem `dados/`, `dados_uf/`, `referencias_imdc3/`, `v4_lib.py`, `comp_lib.py`, `comp_modelos.py`, `uf_lib.py`):
1. suba **`semanal_lib.py`** (pasta `data_imdc_2026/SEMANAL/`);
2. **crie à mão** as pastas **`resultados_semanal`** e **`resultados_uf_prox`** no Drive (assim, duas sessões abertas ao mesmo tempo não criam cópias com o mesmo nome);
3. (opcional, para o Felipe na temporada) a pasta `COMPARACAO_FELIPE/` com `felipe_no_meu_jogo_previsoes.parquet`.

**Retomada:** cada modelo × rodada vira um arquivo assim que termina (gravado num `.tmp` e renomeado, então uma queda no meio nunca deixa arquivo pela metade). Se a sessão cair (internet, energia, limite do Colab), **rode de novo todas as células**: o que já existe é pulado. A seleção de variáveis também grava o estado a cada passo.

| Etapa | O que faz | Tempo (CPU High-RAM) |
|---|---|---|
| 3 · seleção | seleção para trás por grupo de variáveis, **só na afinação** (grava a cada passo) | 30–60 min |
| 4 · modelos | referências, Felipe (XGB/LGBM/CatBoost, ponto e quantis), GBMs (só casos / todas / selecionadas; LightGBM, XGBoost, CatBoost), ARIMA_Fourier, VAR, AutoARIMA | 2–4 h |


In [ ]:
NOTEBOOK = 'semanal_cpu'
PACOTES = 'catboost'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/sem', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'dados', 'dados_uf',
                 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_SEM = os.path.join(PASTA_DRIVE, 'resultados_semanal')       # previsões semanais (1 arquivo por modelo × rodada)
PASTA_PROX = os.path.join(PASTA_DRIVE, 'resultados_uf_prox')      # próximos passos da temporada (lida pelo CompUF4/Comp5)
PASTA_AVAL = os.path.join(PASTA_DRIVE, 'resultados_semanal_avaliacao')
PASTA_RES = os.path.join(PASTA_DRIVE, 'resultados_uf')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (PASTA_SEM, PASTA_PROX, PASTA_AVAL):
    os.makedirs(p, exist_ok=True)
ARQ_SEL = os.path.join(PASTA_SEM, 'selecao_grupos.json')
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S
for m in (V, L, CM, U, S):
    importlib.reload(m)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
except ImportError:
    torch = None
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False                         # True refaz o que já existe (normalmente: False)


In [ ]:
# @title 2 · Rodadas (8 temporadas + a do Felipe)
RODADAS = S.rodadas(ctx.Du, felipe=True)
D = ctx.Du
pd.DataFrame([dict(rodada=k, grupo=S.grupo_rodada(k), treino_ate=str(pd.Timestamp(D.datas[R['c']]).date()),
                   primeira_semana=str(pd.Timestamp(D.datas[R['t_ini']]).date()),
                   ultima_semana=str(pd.Timestamp(D.datas[R['t_fim']]).date())) for k, R in RODADAS.items()])


## 3 · Seleção de variáveis (modelos que rodam melhor com variáveis curadas)
Começa com **todos** os grupos do `uf_lib` (clima, oceano, vizinhos, imunidade física, espalhamento, chik, Afya, estáticas, dispersão, SEAS5) e tira, um por vez, o grupo cuja retirada mais melhora o erro da mediana (log, h = 1–4) nas rodadas de **afinação**. Para quando nenhuma retirada melhora mais que 0,2%. O resultado (`selecao_grupos.json`) é usado pelos modelos `_sel` aqui e no notebook 2.


In [ ]:
# @title 3 · Seleção (retoma de onde parou)
GRUPOS_SEL = S.selecionar_grupos(ctx, RODADAS, ARQ_SEL)
print('Grupos escolhidos:', GRUPOS_SEL)
est = json.load(open(ARQ_SEL))
pd.DataFrame(est['historico'])


## 4 · Modelos semanais (CPU)
- **Todas as variáveis:** GBM_*_todas (os modelos de árvore aguentam todas).
- **Variáveis curadas:** GBM_lgb_sel (grupos da etapa 3), nas versões crescimento e âncora.
- **Só casos:** GBM_lgb_casos, ARIMA, VAR, referências e os modelos do Felipe (as variáveis dele).


In [ ]:
# @title 4 · Lista (ordem = prioridade)
MODELOS = [S.PersistenciaQ(), S.ClimatologiaSem(),
           S.FelipeSemanal('xgb'), S.FelipeSemanal('lgbm'), S.FelipeSemanal('cat'), S.FelipeSemanal('xgb', quantis=True),
           S.GBMSemanal('lgb', 'todas'), S.GBMSemanal('lgb', 'sel', arq_sel=ARQ_SEL), S.GBMSemanal('lgb', 'casos'),
           S.ARIMASemanal(), S.VARSemanal(),
           S.GBMSemanal('lgb', 'sel', alvo='ancora', arq_sel=ARQ_SEL),
           S.GBMSemanal('xgb', 'todas'), S.GBMSemanal('cat', 'todas'),
           S.ARIMASemanal(auto=True)]
[m.nome for m in MODELOS]


In [ ]:
# @title Rodar (retoma de onde parou; um erro não interrompe o resto)
erros = []
for m in MODELOS:
    erros += S.rodar_semanal(ctx, m, RODADAS, PASTA_SEM, refazer=REFAZER)
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'rodada', 'erro']).to_csv(os.path.join(PASTA_SEM, f'erros_{NOTEBOOK}.csv'), index=False)


In [ ]:
# @title Progresso (● = feito)
feitos = {os.path.basename(f)[:-8] for f in glob.glob(os.path.join(PASTA_SEM, '*.parquet'))}
pd.DataFrame({m.nome: {r: '●' if f'{m.nome}__{r}' in feitos else '·' for r in RODADAS} for m in MODELOS}).T
